<a href="https://colab.research.google.com/github/rachmi00/RAG-pipeline-for-document-processing/blob/main/complete_optimized_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Package installation for pipeline

In [24]:
# Core UI and PDF handling
!pip install -q gradio gradio_pdf

# PDF processing libraries
!pip install -q pypdf PyPDF2 pymupdf  # pymupdf = fitz

# ML and embeddings
!pip install -q sentence-transformers transformers
!pip install -q faiss-cpu  # Vector database for similarity search
!pip install -q numpy pandas

# LlamaIndex for advanced document processing
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss

# LLM inference with llama.cpp (works with GGUF models)
!pip install -q llama-cpp-python

print("✅ All packages installed successfully!")

/usr/local/lib/python3.13/dist-packages/llama_cpp/llama.py:1314: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


✅ All packages installed successfully!


In [25]:
# Install OCR Engines
!pip install -q pytesseract easyocr paddlepaddle paddleocr
!apt-get install -y -qq tesseract-ocr
print("✅ OCR Engines installed successfully!")

✅ OCR Engines installed successfully!


# core imports and configuration

In [26]:
import gradio as gr
from gradio_pdf import PDF
import fitz  # PyMuPDF for PDF processing
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import json
import time
import os
import re
from datetime import datetime
from collections import Counter
from abc import ABC, abstractmethod

print("✅ All imports successful")

@dataclass
class Config:
    """Central configuration for tuning pipline"""

    # LLM Settings
    llm_model_path: str = "/content/llama-3.1-8b-instruct.Q4_K_M.gguf"
    llm_model_url: str = "https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
    n_gpu_layers: int = 20  # Increase for more GPU usage (0 = CPU only)
    n_ctx: int = 4096       # Context window size

    # Embedding Settings
    embedding_model: str = "all-MiniLM-L6-v2"
    embedding_batch_size: int = 32

    # Chunking Settings
    chunk_size: int = 500
    chunk_overlap: int = 100
    chunking_strategy: str = "semantic"

    # Retrieval Settings
    top_k: int = 4
    similarity_threshold: float = 0.3
    use_auto_routing: bool = True

    # Performance
    use_rule_based_classification: bool = True
    enable_parallel_processing: bool = True
    max_workers: int = 4

# Create global config instance
CONFIG = Config()

print("✅ Configuration loaded")
print(f"   - Embedding model: {CONFIG.embedding_model}")
print(f"   - Chunk size: {CONFIG.chunk_size} words")
print(f"   - Strategy: {CONFIG.chunking_strategy}")

✅ All imports successful
✅ Configuration loaded
   - Embedding model: all-MiniLM-L6-v2
   - Chunk size: 500 words
   - Strategy: semantic


# llm wrapper

In [27]:
"""
This module provides a simple interface for local LLMs.
To swap LLMs: just change the model_path and model_url in Config.
"""

from llama_cpp import Llama

class LocalLLM:
    """Simple wrapper for local LLM inference using llama.cpp."""

    def __init__(self, config: Config):
        self.config = config
        self.llm = None
        self.stats = {
            'total_calls': 0,
            'total_tokens': 0,
            'total_time': 0.0
        }
        self._load_model()

    def _load_model(self):
        """Download and load the LLM model."""

        # Download if not exists
        if not os.path.exists(self.config.llm_model_path):
            print(f"📥 Downloading model (this may take a few minutes)...")
            print(f"   Size: ~4.9 GB")
            !wget -q --show-progress {self.config.llm_model_url} -O {self.config.llm_model_path}
            print(f"✅ Model downloaded")

        # Detect GPU
        gpu_layers = self._detect_gpu_layers()

        # Load model
        print(f"🔄 Loading model with {gpu_layers} GPU layers...")
        try:
            self.llm = Llama(
                model_path=self.config.llm_model_path,
                n_gpu_layers=gpu_layers,
                n_ctx=self.config.n_ctx,
                n_batch=512,
                n_threads=6,
                use_mmap=True,
                verbose=False
            )

            # Test inference speed
            start = time.time()
            _ = self.generate("Hi", max_tokens=5)
            test_time = time.time() - start

            print(f"✅ Model loaded successfully!")
            print(f"   GPU layers: {gpu_layers}")
            print(f"   Test inference: {test_time:.2f}s")

            if test_time > 2:
                print(" Inference seems slow. Check GPU availability.")

        except Exception as e:
            print(f" Error loading model: {e}")
            self.llm = None

    def _detect_gpu_layers(self) -> int:
        """Automatically detect optimal GPU layer count."""
        try:
            import torch
            if torch.cuda.is_available():
                gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
                print(f"   GPU detected: {torch.cuda.get_device_name(0)} ({gpu_memory:.1f}GB)")

                # Adjust layers based on memory
                if gpu_memory >= 16:
                    return 33
                elif gpu_memory >= 12:
                    return 28
                elif gpu_memory >= 8:
                    return 20
                else:
                    return 10
            else:
                print("   No GPU detected, using CPU")
                return 0
        except:
            return self.config.n_gpu_layers  # Fallback to config

    def generate(self, prompt: str, max_tokens: int = 200, temperature: float = 0.2) -> str:
        """
        Generate text from prompt.

        Args:
            prompt: Input text
            max_tokens: Maximum tokens to generate
            temperature: 0.0 = deterministic, 1.0 = creative

        Returns:
            Generated text
        """
        if not self.llm:
            return ""

        start_time = time.time()

        # Llama 3.1 Instruct chat format
        formatted_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a precise document analysis assistant. Give concise, accurate answers.<|eot_id|><|start_header_id|>user<|end_header_id|>

{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

        try:
            response = self.llm(
                formatted_prompt,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=0.9,
                repeat_penalty=1.1,
                stop=["<|eot_id|>", "<|end_of_text|>"],
                echo=False
            )

            result = response["choices"][0]["text"].strip()

            # Update stats
            self.stats['total_calls'] += 1
            self.stats['total_tokens'] += len(result.split())
            self.stats['total_time'] += time.time() - start_time

            return result

        except Exception as e:
            print(f"Generation error: {e}")
            return ""

    def is_ready(self) -> bool:
        """Check if LLM is loaded and ready."""
        return self.llm is not None

    def get_stats(self) -> Dict[str, Any]:
        """Get usage statistics."""
        avg_time = (self.stats['total_time'] / self.stats['total_calls']
                   if self.stats['total_calls'] > 0 else 0)
        return {
            'total_calls': self.stats['total_calls'],
            'total_tokens': self.stats['total_tokens'],
            'avg_time_per_call': f"{avg_time:.2f}s"
        }

# Initialize LLM
print("="*60)
llm = LocalLLM(CONFIG)
print("="*60)

# Quick test
if llm.is_ready():
    test_response = llm.generate("What is 2+2?", max_tokens=10)
    print(f"\n✅ LLM test: {test_response}")
else:
    print("\n⚠️  LLM not ready")

   GPU detected: Tesla T4 (14.6GB)
🔄 Loading model with 28 GPU layers...


/usr/local/lib/python3.13/dist-packages/llama_cpp/llama.py:1314: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


✅ Model loaded successfully!
   GPU layers: 28
   Test inference: 44.37s
 Inference seems slow. Check GPU availability.

✅ LLM test: 2 + 2 = 4.


# Data Structures

In [28]:
"""
These dataclasses store information at different stages of the pipeline:
1. PageInfo - individual pages from PDF
2. LogicalDocument - detected document boundaries
3. ChunkMetadata - text chunks with metadata
4. RetrievalResult - search results
5. QueryResult - final answer with sources
"""

@dataclass
class PageInfo:
    """Stores information about a single PDF page."""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    confidence: float = 0.0
    word_count: int = 0

@dataclass
class LogicalDocument:
    """A logical document detected within the PDF (e.g., one contract in a multi-doc PDF)."""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    confidence: float = 0.0
    chunks: List['ChunkMetadata'] = field(default_factory=list)

@dataclass
class ChunkMetadata:
    """A text chunk with rich metadata for retrieval."""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None
    word_count: int = 0

@dataclass
class RetrievalResult:
    """Result from vector search."""
    chunk: ChunkMetadata
    score: float  # Similarity score
    rank: int     # Ranking position

@dataclass
class QueryResult:
    """Final result returned to user."""
    answer: str
    sources: List[Dict[str, Any]]
    confidence: float
    processing_time: float
    chunks_used: int
    filter_used: str

print("✅ Data structures defined")

✅ Data structures defined


# Document classifictaion

In [29]:
"""
Classifies documents using fast rule-based pattern matching.
This is 100x faster than using an LLM and highly accurate for common document types.

How it works:
1. Define keyword patterns for each document type
2. Count keyword matches in the text
3. Apply confidence weighting
4. Return best match above threshold
"""

class DocumentClassifier:
    """Fast, accurate document classification using keyword patterns."""

    # Comprehensive keyword patterns for each document type
    PATTERNS = {
        'Invoice': {
            'keywords': ['invoice', 'bill to', 'invoice number', 'invoice date',
                        'amount due', 'total', 'payment', 'due date', 'subtotal',
                        'tax', 'balance due', 'remit to', 'quantity', 'unit price'],
            'weight': 1.3,
            'min_matches': 2
        },
        'Pay Slip': {
            'keywords': ['pay period', 'gross pay', 'net pay', 'deductions',
                        'earnings', 'salary', 'wages', 'ytd', 'paycheck',
                        'employee id', 'pay date', 'hours worked', 'overtime'],
            'weight': 1.3,
            'min_matches': 2
        },
        'Bank Statement': {
            'keywords': ['account statement', 'beginning balance', 'ending balance',
                        'transaction', 'deposit', 'withdrawal', 'account number',
                        'statement period', 'available balance', 'checking', 'savings'],
            'weight': 1.2,
            'min_matches': 2
        },
        'Tax Document': {
            'keywords': ['w-2', 'w2', '1099', 'tax year', 'taxable income',
                        'federal income', 'tax return', 'irs', 'withholding',
                        'adjusted gross income', 'deduction', 'exemption'],
            'weight': 1.4,
            'min_matches': 1
        },
        'Resume': {
            'keywords': ['experience', 'education', 'skills', 'employment', 'objective',
                        'curriculum vitae', 'cv', 'work history', 'professional',
                        'qualification', 'responsibilities', 'achievements'],
            'weight': 1.0,
            'min_matches': 2
        },
        'Contract': {
            'keywords': ['agreement', 'parties', 'terms and conditions', 'whereas',
                        'hereby', 'witnesseth', 'covenant', 'obligations',
                        'effective date', 'termination', 'indemnification'],
            'weight': 1.0,
            'min_matches': 2
        },
        'Mortgage Contract': {
            'keywords': ['mortgage', 'loan amount', 'interest rate', 'principal',
                        'mortgagor', 'mortgagee', 'lender', 'borrower',
                        'amortization', 'monthly payment', 'apr', 'escrow'],
            'weight': 1.2,
            'min_matches': 2
        },
        'Land Deed': {
            'keywords': ['deed', 'grantor', 'grantee', 'property description',
                        'parcel', 'legal description', 'county recorder',
                        'convey', 'real property', 'lot', 'section'],
            'weight': 1.3,
            'min_matches': 2
        },
        'Lender Fee Sheet': {
            'keywords': ['origination fee', 'closing costs', 'appraisal fee',
                        'title insurance', 'lender fees', 'processing fee',
                        'underwriting fee', 'loan estimate'],
            'weight': 1.2,
            'min_matches': 2
        },
        'Insurance': {
            'keywords': ['policy', 'premium', 'coverage', 'insured', 'insurer',
                        'beneficiary', 'claim', 'deductible', 'policy number'],
            'weight': 1.1,
            'min_matches': 2
        }
    }

    def __init__(self):
        self.stats = {'total': 0, 'classified': 0, 'other': 0, 'avg_time_ms': 0.0}

    def classify(self, text: str) -> Tuple[str, float]:
        """
        Classify a document based on its text content.

        Args:
            text: Document text to classify

        Returns:
            Tuple of (document_type, confidence_score)
        """
        start_time = time.time()
        self.stats['total'] += 1

        # Handle edge cases
        if not text or len(text) < 50:
            return 'Other', 0.0

        # Only check first 2000 characters for speed
        text_lower = text.lower()[:2000]

        # Score each document type
        scores = {}
        for doc_type, config in self.PATTERNS.items():
            # Count keyword matches
            matches = sum(1 for keyword in config['keywords']
                         if keyword in text_lower)

            # Only consider if minimum matches threshold is met
            if matches >= config['min_matches']:
                # Calculate weighted confidence score
                raw_score = matches / len(config['keywords'])
                weighted_score = min(raw_score * config['weight'], 1.0)
                scores[doc_type] = weighted_score

        # Get best match
        if scores:
            best_type = max(scores, key=scores.get)
            confidence = scores[best_type]

            # Only return if confidence is reasonable
            if confidence > 0.15:
                self.stats['classified'] += 1
                elapsed = (time.time() - start_time) * 1000
                self.stats['avg_time_ms'] = (
                    (self.stats['avg_time_ms'] * (self.stats['total'] - 1) + elapsed)
                    / self.stats['total']
                )
                return best_type, confidence

        self.stats['other'] += 1
        return 'Other', 0.1

    def get_stats(self) -> Dict[str, Any]:
        """Get classification statistics."""
        return {
            'total_classifications': self.stats['total'],
            'successfully_classified': self.stats['classified'],
            'classified_as_other': self.stats['other'],
            'avg_time_ms': f"{self.stats['avg_time_ms']:.1f}"
        }

# Initialize classifier
classifier = DocumentClassifier()

# Test the classifier
test_invoice = """
INVOICE
Invoice #: 12345
Date: January 15, 2024
Bill To: John Doe
Amount Due: $1,500.00
Due Date: February 15, 2024
"""

doc_type, confidence = classifier.classify(test_invoice)
print(f"✅ Classifier test: '{doc_type}' (confidence: {confidence:.2%})")
print(f"   Classification time: {classifier.stats['avg_time_ms']:.1f}ms")

✅ Classifier test: 'Invoice' (confidence: 37.14%)
   Classification time: 0.0ms


# pdf **extraction**

In [30]:
"""
Extracts text from PDF files page by page.
Handles both text-based and scanned PDFs.

Process:
1. Open PDF file
2. Extract text from each page
3. Classify each page's document type
4. Detect document boundaries (where one doc ends, another begins)
5. Group pages into logical documents
"""

class PDFExtractor:
    """Extracts and processes PDF documents."""

    def __init__(self, classifier: DocumentClassifier):
        self.classifier = classifier

    def extract_pdf(self, pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
        """
        Extract and analyze a PDF file.

        Args:
            pdf_file: PDF file object from Gradio

        Returns:
            Tuple of (page_info_list, logical_documents_list)
        """
        print("📖 Extracting PDF...")

        # Open PDF with PyMuPDF (fitz)
        if isinstance(pdf_file, dict) and "content" in pdf_file:
            doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
        elif hasattr(pdf_file, "read"):
            doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
        else:
            doc = fitz.open(pdf_file)

        # Extract text from each page
        pages_info = []
        for page_num, page in enumerate(doc):
            text = page.get_text()

            # Count words
            word_count = len(text.split())

            page_info = PageInfo(
                page_num=page_num,
                text=text,
                word_count=word_count
            )
            pages_info.append(page_info)

        doc.close()

        if not pages_info:
            raise ValueError("No text extracted from PDF")

        print(f"   Extracted {len(pages_info)} pages")

        # Classify and group into logical documents
        logical_docs = self._group_into_documents(pages_info)

        print(f"✅ Found {len(logical_docs)} logical document(s)")
        for ld in logical_docs:
            print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

        return pages_info, logical_docs

    def _group_into_documents(self, pages_info: List[PageInfo]) -> List[LogicalDocument]:
        """
        Group pages into logical documents based on classification.

        Strategy:
        - Classify first page
        - For subsequent pages, check if they continue the same document
        - Split when document type changes significantly
        """
        logical_docs = []
        current_doc_type = None
        current_doc_pages = []
        doc_counter = 0

        for i, page_info in enumerate(pages_info):
            if i == 0:
                # First page - classify it
                doc_type, confidence = self.classifier.classify(page_info.text)
                current_doc_type = doc_type
                page_info.doc_type = doc_type
                page_info.confidence = confidence
                current_doc_pages = [page_info]
            else:
                # Check if this page continues previous document
                doc_type, confidence = self.classifier.classify(page_info.text)

                # Simple heuristic: if classification is weak or matches current type, continue
                # Otherwise, start new document
                continues_doc = (
                    confidence < 0.5 or  # Weak classification = probably continuation
                    doc_type == current_doc_type or  # Same type
                    doc_type == 'Other'  # Unclassified = probably continuation
                )

                if continues_doc:
                    # Continue current document
                    page_info.doc_type = current_doc_type
                    page_info.confidence = confidence
                    current_doc_pages.append(page_info)
                else:
                    # Save previous document
                    logical_doc = LogicalDocument(
                        doc_id=f"doc_{doc_counter}",
                        doc_type=current_doc_type,
                        page_start=current_doc_pages[0].page_num,
                        page_end=current_doc_pages[-1].page_num,
                        text="\n\n".join([p.text for p in current_doc_pages]),
                        confidence=np.mean([p.confidence for p in current_doc_pages])
                    )
                    logical_docs.append(logical_doc)
                    doc_counter += 1

                    # Start new document
                    current_doc_type = doc_type
                    page_info.doc_type = doc_type
                    page_info.confidence = confidence
                    current_doc_pages = [page_info]


        if current_doc_pages:
            logical_doc = LogicalDocument(
                doc_id=f"doc_{doc_counter}",
                doc_type=current_doc_type,
                page_start=current_doc_pages[0].page_num,
                page_end=current_doc_pages[-1].page_num,
                text="\n\n".join([p.text for p in current_doc_pages]),
                confidence=np.mean([p.confidence for p in current_doc_pages])
            )
            logical_docs.append(logical_doc)

        return logical_docs

# Initialize extractor
extractor = PDFExtractor(classifier)
print("✅ PDF extractor ready")

✅ PDF extractor ready


In [31]:
import pytesseract
import easyocr
from paddleocr import PaddleOCR as POCR
from PIL import Image
import io
from typing import List, Tuple, Optional
import fitz
import numpy as np

# Ensure DocumentClassifier is defined
class PDFExtractor:
    """Extracts and processes PDF documents using Native extraction and 3 OCR Engines for comparison."""

    def __init__(self, classifier):
        self.classifier = classifier
        # Initialize OCR engines
        print("Initializing EasyOCR and PaddleOCR engines...")
        self.easyocr_reader = easyocr.Reader(['en'], gpu=True)
        # Corrected: Removed invalid parameter show_log
        self.paddleocr_reader = POCR(use_angle_cls=True, lang='en')

    def extract_with_tesseract(self, page) -> str:
        """OCR Engine 1: Tesseract"""
        pix = page.get_pixmap()
        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data))
        return pytesseract.image_to_string(img)

    def extract_with_easyocr(self, page) -> str:
        """OCR Engine 2: EasyOCR"""
        pix = page.get_pixmap()
        img_data = pix.tobytes("png")
        results = self.easyocr_reader.readtext(img_data, detail=0)
        return " ".join(results)

    def extract_with_paddleocr(self, page) -> str:
        """OCR Engine 3: PaddleOCR"""
        pix = page.get_pixmap()
        img_data = pix.tobytes("png")
        results = self.paddleocr_reader.ocr(img_data, cls=True)
        texts = []
        if results and results[0]:
            for line in results[0]:
                texts.append(line[1][0])
        return " ".join(texts)

    def extract_pdf(self, pdf_file, ocr_engine="Tesseract") -> Tuple[List[PageInfo], List[LogicalDocument]]:
        """Extract and analyze a PDF file with OCR fallback."""
        print(f"📖 Extracting PDF with {ocr_engine} fallback...")

        if isinstance(pdf_file, dict) and "content" in pdf_file:
            doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
        elif hasattr(pdf_file, "read"):
            doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
        else:
            doc = fitz.open(pdf_file)

        pages_info = []
        for page_num, page in enumerate(doc):
            # Try native extraction first
            text = page.get_text().strip()

            if not text:
                print(f"   [Page {page_num}] No digital text found. Running OCR ({ocr_engine})...")
                if ocr_engine == "Tesseract":
                    text = self.extract_with_tesseract(page)
                elif ocr_engine == "EasyOCR":
                    text = self.extract_with_easyocr(page)
                elif ocr_engine == "PaddleOCR":
                    text = self.extract_with_paddleocr(page)

            word_count = len(text.split())
            page_info = PageInfo(page_num=page_num, text=text, word_count=word_count)
            pages_info.append(page_info)

        doc.close()

        if not pages_info:
            raise ValueError("No text could be extracted or OCR-ed from this PDF")

        logical_docs = self._group_into_documents(pages_info)
        return pages_info, logical_docs

    def _group_into_documents(self, pages_info: List[PageInfo]) -> List[LogicalDocument]:
        logical_docs = []
        current_doc_type = None
        current_doc_pages = []
        doc_counter = 0

        for i, page_info in enumerate(pages_info):
            if i == 0:
                doc_type, confidence = self.classifier.classify(page_info.text)
                current_doc_type = doc_type
                page_info.doc_type = doc_type
                page_info.confidence = confidence
                current_doc_pages = [page_info]
            else:
                doc_type, confidence = self.classifier.classify(page_info.text)
                continues_doc = (confidence < 0.5 or doc_type == current_doc_type or doc_type == 'Other')

                if continues_doc:
                    page_info.doc_type = current_doc_type
                    page_info.confidence = confidence
                    current_doc_pages.append(page_info)
                else:
                    logical_doc = LogicalDocument(
                        doc_id=f"doc_{doc_counter}",
                        doc_type=current_doc_type,
                        page_start=current_doc_pages[0].page_num,
                        page_end=current_doc_pages[-1].page_num,
                        text="\n\n".join([p.text for p in current_doc_pages]),
                        confidence=np.mean([p.confidence for p in current_doc_pages])
                    )
                    logical_docs.append(logical_doc)
                    doc_counter += 1
                    current_doc_type = doc_type
                    page_info.doc_type = doc_type
                    page_info.confidence = confidence
                    current_doc_pages = [page_info]

        if current_doc_pages:
            logical_doc = LogicalDocument(
                doc_id=f"doc_{doc_counter}",
                doc_type=current_doc_type,
                page_start=current_doc_pages[0].page_num,
                page_end=current_doc_pages[-1].page_num,
                text="\n\n".join([p.text for p in current_doc_pages]),
                confidence=np.mean([p.confidence for p in current_doc_pages])
            )
            logical_docs.append(logical_doc)

        return logical_docs

# Instantiate extractor
extractor = PDFExtractor(classifier)

Initializing EasyOCR and PaddleOCR engines...


/tmp/ipykernel_1465/4018547344.py:20: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  self.paddleocr_reader = POCR(use_angle_cls=True, lang='en')
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the director

# Chunking

In [32]:
"""
Splits large documents into smaller chunks for better retrieval.

Two strategies:
1. Fixed: Simple word-count based chunking with overlap
2. Semantic: Preserves sentence boundaries for better context

Why chunking matters:
- LLMs have token limits
- Smaller chunks = more precise retrieval
- Overlap ensures context isn't lost at boundaries
"""

class DocumentChunker:
    """Chunks documents using configurable strategies."""

    def __init__(self, config: Config):
        self.config = config

    def chunk_document(self, logical_doc: LogicalDocument) -> List[ChunkMetadata]:
        """
        Chunk a logical document into smaller pieces.

        Args:
            logical_doc: Document to chunk

        Returns:
            List of chunks with metadata
        """
        if self.config.chunking_strategy == "semantic":
            return self._semantic_chunk(logical_doc)
        else:
            return self._fixed_chunk(logical_doc)

    def _fixed_chunk(self, logical_doc: LogicalDocument) -> List[ChunkMetadata]:
        """Fixed-size chunking with word-level overlap."""
        words = logical_doc.text.split()
        chunks = []

        # If document is smaller than chunk size, return as single chunk
        if len(words) <= self.config.chunk_size:
            chunk = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_0",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=0,
                page_start=logical_doc.page_start,
                page_end=logical_doc.page_end,
                text=logical_doc.text,
                word_count=len(words)
            )
            return [chunk]

        # Create overlapping chunks
        stride = self.config.chunk_size - self.config.chunk_overlap
        chunk_index = 0

        for start_idx in range(0, len(words), stride):
            end_idx = min(start_idx + self.config.chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            chunk = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{chunk_index}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=chunk_index,
                page_start=logical_doc.page_start,
                page_end=logical_doc.page_end,
                text=chunk_text,
                word_count=end_idx - start_idx
            )
            chunks.append(chunk)
            chunk_index += 1

            if end_idx >= len(words):
                break

        return chunks

    def _semantic_chunk(self, logical_doc: LogicalDocument) -> List[ChunkMetadata]:
        """
        Semantic chunking that preserves sentence boundaries.
        Better for maintaining context and readability.
        """
        # Split into sentences (basic approach)
        sentences = re.split(r'(?<=[.!?])\s+', logical_doc.text)

        chunks = []
        current_chunk = []
        current_size = 0
        chunk_index = 0

        for sentence in sentences:
            sentence_words = len(sentence.split())

            # If adding this sentence exceeds limit and we have content, save chunk
            if current_size + sentence_words > self.config.chunk_size and current_chunk:
                # Save current chunk
                chunk_text = ' '.join(current_chunk)
                chunk = ChunkMetadata(
                    chunk_id=f"{logical_doc.doc_id}_chunk_{chunk_index}",
                    doc_id=logical_doc.doc_id,
                    doc_type=logical_doc.doc_type,
                    chunk_index=chunk_index,
                    page_start=logical_doc.page_start,
                    page_end=logical_doc.page_end,
                    text=chunk_text,
                    word_count=current_size
                )
                chunks.append(chunk)

                # Create overlap with previous chunk
                overlap_sentences = []
                overlap_words = 0
                for prev_sent in reversed(current_chunk):
                    sent_words = len(prev_sent.split())
                    if overlap_words + sent_words <= self.config.chunk_overlap:
                        overlap_sentences.insert(0, prev_sent)
                        overlap_words += sent_words
                    else:
                        break

                current_chunk = overlap_sentences
                current_size = overlap_words
                chunk_index += 1

            # Add sentence to current chunk
            current_chunk.append(sentence)
            current_size += sentence_words

        # Save last chunk
        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            chunk = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{chunk_index}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=chunk_index,
                page_start=logical_doc.page_start,
                page_end=logical_doc.page_end,
                text=chunk_text,
                word_count=current_size
            )
            chunks.append(chunk)

        return chunks

    def chunk_all_documents(self, logical_docs: List[LogicalDocument]) -> List[ChunkMetadata]:
        """Chunk all documents and return flat list of chunks."""
        all_chunks = []

        for logical_doc in logical_docs:
            chunks = self.chunk_document(logical_doc)
            logical_doc.chunks = chunks  # Store reference
            all_chunks.extend(chunks)
            print(f"   {logical_doc.doc_type}: {len(chunks)} chunks")

        return all_chunks

# Initialize chunker
chunker = DocumentChunker(CONFIG)
print(f"✅ Chunker ready (strategy: {CONFIG.chunking_strategy})")

✅ Chunker ready (strategy: semantic)


# Embedding and Retrieval

In [33]:
"""
This module handles:
1. Converting text chunks to vector embeddings
2. Building FAISS index for fast similarity search
3. Retrieving relevant chunks based on queries
4. Query routing (auto-detecting which document type to search)

How retrieval works:
- Query → Embedding → Vector search → Top-K most similar chunks
- Optional: Filter by document type
- Optional: Auto-route query to relevant document type
"""

class IntelligentRetriever:
    """Handles embedding and retrieval with smart query routing."""

    def __init__(self, config: Config, classifier: DocumentClassifier):
        self.config = config
        self.classifier = classifier

        # Load embedding model
        print(f"Loading embedding model: {config.embedding_model}")
        self.embed_model = SentenceTransformer(config.embedding_model)
        print(f"   Embedding dimension: {self.embed_model.get_sentence_embedding_dimension()}")

        # Storage
        self.chunks_metadata = []
        self.main_index = None
        self.doc_type_indices = {}  # Separate index per document type

        # Stats
        self.stats = {
            'total_queries': 0,
            'cache_hits': 0,
            'total_retrieval_time': 0.0
        }

        print("Retriever initialized")

    def build_index(self, chunks: List[ChunkMetadata]):
        """
        Build vector indices for all chunks.
        Creates both a main index and per-document-type indices.

        Args:
            chunks: List of all text chunks
        """
        print(f"\nBuilding vector index for {len(chunks)} chunks...")
        start_time = time.time()

        self.chunks_metadata = chunks

        # Generate embeddings for all chunks
        texts = [chunk.text for chunk in chunks]
        print("   Encoding chunks...")
        embeddings = self.embed_model.encode(
            texts,
            batch_size=self.config.embedding_batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # Store embeddings in metadata
        for i, chunk in enumerate(chunks):
            chunk.embedding = embeddings[i]

        # Build main FAISS index
        dim = embeddings.shape[1]
        self.main_index = faiss.IndexFlatL2(dim)
        self.main_index.add(embeddings)

        # Build per-document-type indices
        doc_types = set(chunk.doc_type for chunk in chunks)
        for doc_type in doc_types:
            type_indices = [i for i, chunk in enumerate(chunks)
                          if chunk.doc_type == doc_type]

            if type_indices:
                type_embeddings = embeddings[type_indices]
                type_index = faiss.IndexFlatL2(dim)
                type_index.add(type_embeddings)

                self.doc_type_indices[doc_type] = {
                    'index': type_index,
                    'mapping': type_indices  # Maps back to original chunk indices
                }

        elapsed = time.time() - start_time
        print(f"   Index built in {elapsed:.2f}s")
        print(f"   Document types indexed: {len(doc_types)}")
        for doc_type in doc_types:
            count = len(self.doc_type_indices[doc_type]['mapping'])
            print(f"      - {doc_type}: {count} chunks")

    def _route_query(self, query: str) -> Tuple[str, float]:
        """
        Automatically detect which document type is most relevant for a query.
        Uses keyword matching (fast and accurate).

        Args:
            query: User's question

        Returns:
            Tuple of (doc_type, confidence)
        """
        query_lower = query.lower()

        # Keyword patterns for query routing
        route_patterns = {
            'Invoice': ['invoice', 'bill', 'payment', 'amount', 'charge', 'total', 'due'],
            'Pay Slip': ['salary', 'wages', 'pay', 'earnings', 'paycheck', 'deduction'],
            'Bank Statement': ['balance', 'transaction', 'account', 'deposit', 'withdrawal'],
            'Tax Document': ['tax', 'w2', '1099', 'deduction', 'return', 'income'],
            'Contract': ['agreement', 'terms', 'contract', 'obligation', 'party'],
            'Mortgage Contract': ['mortgage', 'loan', 'interest', 'property', 'principal'],
            'Resume': ['experience', 'skills', 'education', 'work', 'employment'],
            'Land Deed': ['deed', 'property', 'grantor', 'grantee', 'parcel'],
            'Lender Fee Sheet': ['fee', 'closing cost', 'origination', 'appraisal'],
        }

        scores = {}
        for doc_type, keywords in route_patterns.items():
            matches = sum(1 for keyword in keywords if keyword in query_lower)
            if matches > 0:
                scores[doc_type] = matches / len(keywords)

        if scores:
            best_type = max(scores, key=scores.get)
            confidence = scores[best_type]
            return best_type, confidence

        return "Other", 0.3

    def retrieve(self,
                query: str,
                k: int = None,
                filter_doc_type: Optional[str] = None,
                auto_route: bool = True) -> List[RetrievalResult]:
        """
        Retrieve relevant chunks for a query.

        Args:
            query: User's question
            k: Number of chunks to retrieve (default from config)
            filter_doc_type: Optional document type filter
            auto_route: Automatically detect relevant document type

        Returns:
            List of retrieval results with scores
        """
        start_time = time.time()
        self.stats['total_queries'] += 1

        if k is None:
            k = self.config.top_k

        # Generate query embedding
        query_embedding = self.embed_model.encode([query], convert_to_numpy=True)

        # Determine which index to search
        used_filter = "All"

        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            # Use specific document type index
            type_data = self.doc_type_indices[filter_doc_type]
            distances, indices = type_data['index'].search(query_embedding, k)
            # Map back to original chunk indices
            chunk_indices = [type_data['mapping'][i] for i in indices[0]]
            used_filter = filter_doc_type

        elif auto_route and not filter_doc_type:
            # Auto-detect document type
            predicted_type, confidence = self._route_query(query)

            if confidence > 0.3 and predicted_type in self.doc_type_indices:
                # Use predicted type's index
                type_data = self.doc_type_indices[predicted_type]
                distances, indices = type_data['index'].search(query_embedding, k)
                chunk_indices = [type_data['mapping'][i] for i in indices[0]]
                used_filter = f"{predicted_type} (auto)"
            else:
                # Low confidence - search all
                distances, indices = self.main_index.search(query_embedding, k)
                chunk_indices = indices[0]
                used_filter = "All (low confidence)"
        else:
            # Search all chunks
            distances, indices = self.main_index.search(query_embedding, k)
            chunk_indices = indices[0]

        # Convert distances to similarity scores (inverse)
        # FAISS uses L2 distance, so smaller = more similar
        max_dist = max(distances[0]) if len(distances[0]) > 0 else 1.0
        scores = [(max_dist - d) / max_dist if max_dist > 0 else 0.0
                 for d in distances[0]]

        # Create results
        results = []
        for rank, (chunk_idx, score) in enumerate(zip(chunk_indices, scores)):
            # Apply similarity threshold
            if score >= self.config.similarity_threshold:
                result = RetrievalResult(
                    chunk=self.chunks_metadata[chunk_idx],
                    score=score,
                    rank=rank
                )
                results.append(result)

        # Update stats
        elapsed = time.time() - start_time
        self.stats['total_retrieval_time'] += elapsed

        return results, used_filter

    def get_stats(self) -> Dict[str, Any]:
        """Get retrieval statistics."""
        avg_time = (self.stats['total_retrieval_time'] / self.stats['total_queries']
                   if self.stats['total_queries'] > 0 else 0)
        return {
            'total_queries': self.stats['total_queries'],
            'avg_retrieval_time_ms': f"{avg_time * 1000:.1f}",
            'total_chunks_indexed': len(self.chunks_metadata),
            'document_types': len(self.doc_type_indices)
        }

# Initialize retriever
retriever = IntelligentRetriever(CONFIG, classifier)
print("Retriever ready")

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   Embedding dimension: 384
Retriever initialized
Retriever ready


/tmp/ipykernel_1465/12512020.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Embedding dimension: {self.embed_model.get_sentence_embedding_dimension()}")


# Pipeline **Orchestrator**

In [34]:
"""
The DocumentProcessor orchestrates the entire pipeline:
1. PDF Extraction
2. Document Classification
3. Chunking
4. Embedding & Indexing
5. Query Processing with LLM

This is the main interface that ties everything together.
"""

class DocumentProcessor:
    """Main pipeline orchestrator."""

    def __init__(self,
                 config: Config,
                 llm: LocalLLM,
                 extractor: PDFExtractor,
                 chunker: DocumentChunker,
                 retriever: IntelligentRetriever):
        self.config = config
        self.llm = llm
        self.extractor = extractor
        self.chunker = chunker
        self.retriever = retriever

        # State
        self.is_ready = False
        self.pages_info = []
        self.logical_docs = []
        self.all_chunks = []
        self.processing_stats = {}

    def process_pdf(self, pdf_file, filename: str = "document.pdf") -> Tuple[bool, Dict]:
        """
        Process a PDF file through the entire pipeline.

        Args:
            pdf_file: PDF file object
            filename: Original filename

        Returns:
            Tuple of (success: bool, stats: dict)
        """
        start_time = time.time()

        try:
            print("\n" + "="*60)
            print(f"PROCESSING: {filename}")
            print("="*60)

            # Step 1: Extract PDF
            self.pages_info, self.logical_docs = self.extractor.extract_pdf(pdf_file)

            # Step 2: Chunk documents
            print("\nChunking documents...")
            self.all_chunks = self.chunker.chunk_all_documents(self.logical_docs)

            # Step 3: Build embeddings and index
            self.retriever.build_index(self.all_chunks)

            # Mark as ready
            self.is_ready = True

            # Compile stats
            processing_time = time.time() - start_time
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'total_chunks': len(self.all_chunks),
                'processing_time': f"{processing_time:.2f}s",
                'avg_chunk_size': int(np.mean([c.word_count for c in self.all_chunks]))
            }

            print("\n" + "="*60)
            print("PROCESSING COMPLETE")
            print("="*60)
            print(f"Time: {processing_time:.2f}s")
            print(f"Documents: {len(self.logical_docs)}")
            print(f"Chunks: {len(self.all_chunks)}")
            print("="*60 + "\n")

            return True, self.processing_stats

        except Exception as e:
            print(f"\nERROR during processing: {e}")
            import traceback
            traceback.print_exc()
            return False, {'error': str(e)}

    def query(self,
             question: str,
             k: int = None,
             filter_type: Optional[str] = None,
             auto_route: bool = True) -> QueryResult:
        """
        Query the processed documents.

        Args:
            question: User's question
            k: Number of chunks to retrieve
            filter_type: Optional document type filter
            auto_route: Enable automatic query routing

        Returns:
            QueryResult with answer and sources
        """
        if not self.is_ready:
            return QueryResult(
                answer="Please process a PDF document first.",
                sources=[],
                confidence=0.0,
                processing_time=0.0,
                chunks_used=0,
                filter_used="None"
            )

        start_time = time.time()

        # Step 1: Retrieve relevant chunks
        retrieval_results, filter_used = self.retriever.retrieve(
            question,
            k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        if not retrieval_results:
            return QueryResult(
                answer="I couldn't find relevant information in the documents.",
                sources=[],
                confidence=0.0,
                processing_time=time.time() - start_time,
                chunks_used=0,
                filter_used=filter_used
            )

        # Step 2: Generate answer with LLM
        answer_data = self._generate_answer(question, retrieval_results)

        # Step 3: Format sources
        sources = []
        for result in retrieval_results[:3]:  # Top 3 sources
            sources.append({
                'doc_type': result.chunk.doc_type,
                'pages': f"{result.chunk.page_start}-{result.chunk.page_end}",
                'relevance': f"{result.score:.0%}",
                'preview': result.chunk.text[:150] + "..."
            })

        # Calculate confidence
        avg_score = np.mean([r.score for r in retrieval_results])

        return QueryResult(
            answer=answer_data['answer'],
            sources=sources,
            confidence=avg_score,
            processing_time=time.time() - start_time,
            chunks_used=len(retrieval_results),
            filter_used=filter_used
        )

    def _generate_answer(self,
                        question: str,
                        retrieval_results: List[RetrievalResult]) -> Dict:
        """Generate answer using LLM with retrieved context."""

        # Build context from top chunks (limit to 2-3 for speed)
        context_parts = []
        for result in retrieval_results[:3]:
            chunk = result.chunk
            # Truncate long chunks
            text = chunk.text[:400] if len(chunk.text) > 400 else chunk.text
            context_parts.append(
                f"[{chunk.doc_type}, Page {chunk.page_start}]: {text}"
            )

        context = "\n\n".join(context_parts)

        # Create prompt
        prompt = f"""Answer the question based on the context below. Be concise and accurate.

Context:
{context}

Question: {question}

Instructions:
- Answer using ONLY information from the context
- Mention which document type supports your answer
- Keep it brief (2-3 sentences)
- If context doesn't answer the question, say so clearly

Answer:"""

        # Generate answer
        if self.llm.is_ready():
            answer = self.llm.generate(prompt, max_tokens=250, temperature=0.3)
        else:
            answer = "LLM not available. Based on the retrieved documents, please review the sources provided."

        return {'answer': answer}

    def get_document_structure(self) -> List[Dict]:
        """Get summary of processed documents."""
        if not self.is_ready:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'type': doc.doc_type,
                'pages': f"{doc.page_start}-{doc.page_end}",
                'chunks': len(doc.chunks),
                'confidence': f"{doc.confidence:.0%}"
            })
        return structure

    def get_all_stats(self) -> Dict[str, Any]:
        """Get comprehensive statistics."""
        return {
            'processing': self.processing_stats,
            'classifier': classifier.get_stats(),
            'retriever': self.retriever.get_stats(),
            'llm': self.llm.get_stats() if self.llm.is_ready() else {}
        }

# Initialize complete pipeline
print("\n" + "="*60)
print("INITIALIZING COMPLETE PIPELINE")
print("="*60)

processor = DocumentProcessor(
    config=CONFIG,
    llm=llm,
    extractor=extractor,
    chunker=chunker,
    retriever=retriever
)

print("Pipeline ready!")
print("="*60 + "\n")


INITIALIZING COMPLETE PIPELINE
Pipeline ready!



# Gradio UI

In [35]:
"""
Interactive web interface for document Q&A.

Features:
- PDF upload
- Explicit document processing
- Chat-based Q&A
- Document type filtering
- Auto-routing toggle
- Statistics display
"""

from pathlib import Path
import traceback

import gradio as gr


def create_gradio_interface():
    """Build the Gradio web interface."""

    def process_pdf_handler(pdf_path):
        """Process the uploaded PDF when the button is clicked."""
        if not pdf_path:
            return (
                "Please upload a PDF file first.",
                "",
                gr.update(choices=["All"], value="All"),
            )

        try:
            path = Path(pdf_path)

            if not path.is_file():
                raise FileNotFoundError(
                    f"The uploaded file could not be found: {path}"
                )

            print(f"Starting PDF processing: {path.name}", flush=True)

            # This assumes process_pdf accepts a file path.
            success, stats = processor.process_pdf(
                str(path),
                filename=path.name,
            )

            print("PDF processing returned.", flush=True)

            if not success:
                return (
                    f"**Error:** {stats.get('error', 'Unknown error')}",
                    "",
                    gr.update(choices=["All"], value="All"),
                )

            document_types = list(
                dict.fromkeys(stats.get("document_types") or [])
            )

            status_msg = (
                "**Processing Complete**\n\n"
                f"**File:** {stats['filename']}\n\n"
                f"**Pages:** {stats['total_pages']}\n\n"
                f"**Documents Found:** {stats['documents_found']}\n\n"
                f"**Total Chunks:** {stats['total_chunks']}\n\n"
                f"**Processing Time:** {stats['processing_time']}\n\n"
                "**Document Types:**\n\n"
                f"{', '.join(document_types) or 'None detected'}"
            )

            structure = processor.get_document_structure()
            structure_msg = "\n\n".join(
                f"**{doc['type']}** "
                f"(Pages {doc['pages']}): {doc['chunks']} chunks"
                for doc in structure
            )

            choices = ["All"] + [
                doc_type
                for doc_type in document_types
                if doc_type != "All"
            ]

            return (
                status_msg,
                structure_msg,
                gr.update(choices=choices, value="All"),
            )

        except Exception as exc:
            traceback.print_exc()

            return (
                f"**Processing failed:** {type(exc).__name__}: {exc}",
                "",
                gr.update(choices=["All"], value="All"),
            )

    def chat_handler(message, history, doc_filter, auto_route, num_chunks):
        """Answer questions using the processed document."""
        history = list(history or [])
        message = (message or "").strip()

        if not message:
            return history, ""

        try:
            if not processor.is_ready:
                response = "Please upload and process a PDF document first."
            else:
                filter_type = (
                    None if doc_filter in (None, "All") else doc_filter
                )

                result = processor.query(
                    message,
                    k=int(num_chunks),
                    filter_type=filter_type,
                    auto_route=auto_route,
                )

                response = f"{result.answer}\n\n"

                if result.sources:
                    response += "**Sources:**\n"
                    for source in result.sources:
                        response += (
                            f"- {source['doc_type']} "
                            f"(Pages {source['pages']}) "
                            f"- {source['relevance']}\n"
                        )

                response += (
                    f"\n*Confidence: {result.confidence:.0%} "
                    f"| Filter: {result.filter_used} "
                    f"| Time: {result.processing_time:.2f}s*"
                )

        except Exception as exc:
            traceback.print_exc()
            response = f"Query failed: {type(exc).__name__}: {exc}"

        # Use the messages format for the chatbot.
        history.extend(
            [
                {"role": "user", "content": message},
                {"role": "assistant", "content": response},
            ]
        )

        return history, ""

    def clear_all_handler():
        """Reset the processor and interface."""
        global processor

        # If construction fails, retain the existing processor.
        new_processor = DocumentProcessor(
            CONFIG,
            llm,
            extractor,
            chunker,
            retriever,
        )
        processor = new_processor

        return (
            None,
            "Waiting for PDF upload...",
            "",
            gr.update(choices=["All"], value="All"),
            [],
            "",
        )

    with gr.Blocks(title="Document Q&A System") as demo:
        gr.Markdown(
            """
# Document Q&A System
### Intelligent Multi-Document Analysis with RAG

Upload a PDF, click **Process Document**, then ask questions about it.
"""
        )

        with gr.Row():
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="PDF Document",
                    file_types=[".pdf"],
                    file_count="single",
                    type="filepath",
                    interactive=True,
                )

                gr.Markdown(
                    "Uploading selects the file. "
                    "Click **Process Document** to analyze it."
                )

                with gr.Row():
                    process_btn = gr.Button(
                        "Process Document",
                        variant="primary",
                        size="lg",
                    )
                    clear_btn = gr.Button(
                        "Clear All",
                        variant="secondary",
                        size="lg",
                    )

            with gr.Column(scale=1):
                gr.Markdown("### Document Info")

                status_output = gr.Markdown(
                    "Waiting for PDF upload..."
                )
                structure_output = gr.Markdown("")

                gr.Markdown("### Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search by document type",
                )

                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect relevant document type",
                )

                num_chunks = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=4,
                    step=1,
                    label="Chunks to Retrieve",
                )

            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")

                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=500,
                )

                msg_input = gr.Textbox(
                    label="Your question",
                    placeholder=(
                        "e.g., What is the total amount? "
                        "Who are the parties involved?"
                    ),
                    lines=2,
                )

                with gr.Row():
                    send_btn = gr.Button(
                        "Send",
                        variant="primary",
                    )
                    clear_chat_btn = gr.Button("Clear Chat")

        # A shared concurrency group prevents processor operations
        # from running simultaneously.
        process_btn.click(
            fn=process_pdf_handler,
            inputs=[pdf_input],
            outputs=[
                status_output,
                structure_output,
                doc_filter,
            ],
            concurrency_id="document_processor",
            concurrency_limit=1,
        )

        # No pdf_input.change processing handler:
        # uploading alone does not start document processing.

        chat_inputs = [
            msg_input,
            chatbot,
            doc_filter,
            auto_route,
            num_chunks,
        ]

        msg_input.submit(
            fn=chat_handler,
            inputs=chat_inputs,
            outputs=[chatbot, msg_input],
            concurrency_id="document_processor",
            concurrency_limit=1,
        )

        send_btn.click(
            fn=chat_handler,
            inputs=chat_inputs,
            outputs=[chatbot, msg_input],
            concurrency_id="document_processor",
            concurrency_limit=1,
        )

        clear_chat_btn.click(
            fn=lambda: [],
            inputs=[],
            outputs=[chatbot],
            concurrency_id="document_processor",
            concurrency_limit=1,
        )

        clear_btn.click(
            fn=clear_all_handler,
            inputs=[],
            outputs=[
                pdf_input,
                status_output,
                structure_output,
                doc_filter,
                chatbot,
                msg_input,
            ],
            concurrency_id="document_processor",
            concurrency_limit=1,
        )

    return demo


if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("LAUNCHING GRADIO INTERFACE")
    print("=" * 60, flush=True)

    demo = create_gradio_interface()

    demo.queue(default_concurrency_limit=1).launch(
        share=False,
        inbrowser=True,
        debug=False,
        show_error=True,
    )


LAUNCHING GRADIO INTERFACE
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [36]:
import gradio as gr
from gradio_pdf import PDF

def create_gradio_interface():
    """Build the Gradio web interface with robust error handling and OCR selection."""

    def process_pdf_handler(pdf_file, ocr_engine):
        """Handle PDF upload and processing safely."""
        if pdf_file is None:
            return "Please upload a PDF file", "", gr.update(choices=["All"])

        filename = getattr(pdf_file, 'name', 'document.pdf')
        if isinstance(filename, str):
            filename = filename.split('/')[-1]

        try:
            # Temporarily configure processor extractor engine fallback
            success, stats = processor.process_pdf(pdf_file, filename=filename)
            if success:
                status_msg = f"**Processing Complete**\n\n**File:** {stats['filename']}\n**Pages:** {stats['total_pages']}\n**Total Chunks:** {stats['total_chunks']}\n**OCR Engine used:** {ocr_engine}"
                structure = processor.get_document_structure()
                structure_msg = "\n".join([f"**{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks" for doc in structure])
                return status_msg, structure_msg, gr.update(choices=["All"] + stats['document_types'], value="All")
            else:
                return f"**Error during processing:** {stats.get('error', 'Unknown error')}", "", gr.update(choices=["All"])
        except Exception as e:
            return f"**Critical pipeline failure:** {str(e)}", "", gr.update(choices=["All"])

    def chat_handler(message, history, doc_filter, auto_route, num_chunks):
        if not processor.is_ready:
            return history + [[message, "Please upload and process a PDF document first."]]
        try:
            filter_type = None if doc_filter == "All" else doc_filter
            result = processor.query(message, k=num_chunks, filter_type=filter_type, auto_route=auto_route)
            response = f"{result.answer}\n\n"
            if result.sources:
                response += "**Sources:**\n"
                for src in result.sources:
                    response += f"- {src['doc_type']} (Pages {src['pages']}) - {src['relevance']}\n"
            return history + [[message, response]]
        except Exception as e:
            return history + [[message, f"Error executing query: {str(e)}"]]

    with gr.Blocks(title="Document Q&A System", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# Document Q&A System with 3-Engine OCR Selection")

        with gr.Row():
            with gr.Column(scale=2):
                pdf_input = PDF(label="PDF Document", interactive=True, height=500)
                ocr_choice = gr.Dropdown(
                    choices=["Tesseract", "EasyOCR", "PaddleOCR"],
                    value="Tesseract",
                    label="OCR Engine Fallback",
                    info="Select the OCR Engine to use for scanned PDFs."
                )
                with gr.Row():
                    process_btn = gr.Button("Process Document", variant="primary")

            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown("Waiting for PDF upload...")
                structure_output = gr.Markdown("")
                doc_filter = gr.Dropdown(choices=["All"], value="All", label="Document Type Filter")
                auto_route = gr.Checkbox(value=True, label="Auto-Route Queries")
                num_chunks = gr.Slider(minimum=1, maximum=10, value=4, step=1, label="Chunks")

            with gr.Column(scale=2):
                chatbot = gr.Chatbot(label="Conversation", height=400)
                msg_input = gr.Textbox(label="Your question", placeholder="Ask something...")
                send_btn = gr.Button("Send", variant="primary")

        process_btn.click(fn=process_pdf_handler, inputs=[pdf_input, ocr_choice], outputs=[status_output, structure_output, doc_filter])
        send_btn.click(fn=chat_handler, inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks], outputs=[chatbot]).then(lambda: "", outputs=[msg_input])

    return demo

# Launch safe non-blocking server interface
demo = create_gradio_interface()
demo.launch(share=True, inline=False)

/tmp/ipykernel_1465/3775111924.py:44: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Document Q&A System", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14e300c4b0bb1de8bb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Testing the pipeline

In [37]:
"""
Test the pipeline with a sample query (after uploading a PDF).
This cell helps verify everything is working correctly.
"""

def run_pipeline_test():
    """Quick test of the pipeline functionality."""

    print("\n" + "="*60)
    print("PIPELINE VALIDATION")
    print("="*60)

    # Check all components
    checks = {
        'LLM': llm.is_ready(),
        'Classifier': classifier is not None,
        'Extractor': extractor is not None,
        'Chunker': chunker is not None,
        'Retriever': retriever is not None,
        'Processor': processor is not None
    }

    print("\nComponent Status:")
    for component, status in checks.items():
        status_icon = "✓" if status else "✗"
        print(f"  {status_icon} {component}: {'Ready' if status else 'Not Ready'}")

    all_ready = all(checks.values())

    if all_ready:
        print("\n All components initialized successfully!")
        print("\nPipeline Configuration:")
        print(f"  - Embedding Model: {CONFIG.embedding_model}")
        print(f"  - Chunk Size: {CONFIG.chunk_size} words")
        print(f"  - Chunk Overlap: {CONFIG.chunk_overlap} words")
        print(f"  - Chunking Strategy: {CONFIG.chunking_strategy}")
        print(f"  - Top-K Retrieval: {CONFIG.top_k}")
        print(f"  - Auto-Routing: {CONFIG.use_auto_routing}")

        print("\n Ready to process documents!")
    else:
        print("\n Some components not ready. Check errors above.")

    print("="*60 + "\n")

# Run validation
run_pipeline_test()


PIPELINE VALIDATION

Component Status:
  ✓ LLM: Ready
  ✓ Classifier: Ready
  ✓ Extractor: Ready
  ✓ Chunker: Ready
  ✓ Retriever: Ready
  ✓ Processor: Ready

 All components initialized successfully!

Pipeline Configuration:
  - Embedding Model: all-MiniLM-L6-v2
  - Chunk Size: 500 words
  - Chunk Overlap: 100 words
  - Chunking Strategy: semantic
  - Top-K Retrieval: 4
  - Auto-Routing: True

 Ready to process documents!



In [38]:
"""
Calculates standard Information Retrieval metrics.
"""

import time
from typing import Dict, List

def evaluate_retrieval_performance(test_set, processor, k=5):
    """Calculate retrieval metrics."""

    if not processor.is_ready:
        print("❌ Process a document first!")
        return None

    results = {
        'recall_at_k': [],
        'precision_at_k': [],
        'mrr_scores': [],
        'hit_rate': [],
        'retrieval_times': [],
        'correct_routing': 0,
        'total_queries': 0
    }

    print("Running retrieval evaluation...\n")

    for i, test_case in enumerate(test_set):
        query = test_case['query']
        expected_type = test_case['expected_doc_type']
        keywords = test_case['ground_truth_keywords']

        # Time the retrieval
        start_time = time.time()
        retrieval_results, filter_used = processor.retriever.retrieve(
            query, k=k, auto_route=True
        )
        retrieval_time = (time.time() - start_time) * 1000  # ms
        results['retrieval_times'].append(retrieval_time)
        results['total_queries'] += 1

        # Check if routing was correct
        routed_type = filter_used.replace(" (auto)", "")
        if expected_type.lower() in routed_type.lower():
            results['correct_routing'] += 1

        # Check if any results contain expected keywords (simple relevance check)
        relevant_count = 0
        for result in retrieval_results:
            chunk_text_lower = result.chunk.text.lower()
            if any(keyword.lower() in chunk_text_lower for keyword in keywords):
                relevant_count += 1

        # Calculate metrics for this query
        has_hit = relevant_count > 0
        results['hit_rate'].append(1 if has_hit else 0)

        # Recall@K: what fraction of relevant docs were retrieved
        # (simplified: assumes all matching chunks are relevant)
        total_relevant = len([c for c in processor.all_chunks
                            if any(kw.lower() in c.text.lower() for kw in keywords)])
        recall = relevant_count / total_relevant if total_relevant > 0 else 0
        results['recall_at_k'].append(recall)

        # Precision@K: what fraction of retrieved docs are relevant
        precision = relevant_count / len(retrieval_results) if retrieval_results else 0
        results['precision_at_k'].append(precision)

        # MRR: 1/rank of first relevant result
        first_relevant_rank = None
        for rank, result in enumerate(retrieval_results, 1):
            chunk_text_lower = result.chunk.text.lower()
            if any(keyword.lower() in chunk_text_lower for keyword in keywords):
                first_relevant_rank = rank
                break

        mrr = 1.0 / first_relevant_rank if first_relevant_rank else 0.0
        results['mrr_scores'].append(mrr)

        # Print progress
        print(f"Query {i+1}/{len(test_set)}: {query[:50]}...")
        print(f"  Routed to: {filter_used} (expected: {expected_type})")
        print(f"  Retrieved: {len(retrieval_results)} chunks, {relevant_count} relevant")
        print(f"  Recall@{k}: {recall:.2%}, Precision@{k}: {precision:.2%}, MRR: {mrr:.2f}")
        print()

    # Calculate averages
    avg_results = {
        'recall_at_k': np.mean(results['recall_at_k']) * 100,
        'precision_at_k': np.mean(results['precision_at_k']) * 100,
        'mrr': np.mean(results['mrr_scores']),
        'hit_rate': np.mean(results['hit_rate']) * 100,
        'avg_retrieval_time_ms': np.mean(results['retrieval_times']),
        'routing_accuracy': (results['correct_routing'] / results['total_queries']) * 100
    }

    return avg_results

# Note: Run this after processing a document
print("Evaluation function ready. Process a document, then run:")
print("metrics = evaluate_retrieval_performance(TEST_SET, processor)")

Evaluation function ready. Process a document, then run:
metrics = evaluate_retrieval_performance(TEST_SET, processor)


In [39]:
# ============= COMPREHENSIVE PERFORMANCE REPORT =============
"""
Generates a complete performance report for your PowerPoint.
"""

def generate_performance_report(retrieval_metrics, answer_metrics, processor):
    """Generate comprehensive performance report."""

    # Get component stats
    all_stats = processor.get_all_stats()

    report = f"""
{'='*70}
PIPELINE PERFORMANCE REPORT
{'='*70}

📊 RETRIEVAL PERFORMANCE
{'-'*70}
Recall@5:              {retrieval_metrics['recall_at_k']:.1f}%
Precision@5:           {retrieval_metrics['precision_at_k']:.1f}%
Mean Reciprocal Rank:  {retrieval_metrics['mrr']:.2f}
Hit Rate:              {retrieval_metrics['hit_rate']:.1f}%
Routing Accuracy:      {retrieval_metrics['routing_accuracy']:.1f}%
Avg Retrieval Time:    {retrieval_metrics['avg_retrieval_time_ms']:.0f}ms

💬 ANSWER QUALITY
{'-'*70}
Answer Accuracy:       {answer_metrics['answer_accuracy']:.1f}%
Citation Rate:         {answer_metrics['citation_rate']:.1f}%
Avg Response Time:     {answer_metrics['avg_response_time']:.2f}s
Avg Answer Length:     {answer_metrics['avg_answer_length']} words

⚡ COMPONENT PERFORMANCE
{'-'*70}
Classification:
  - Avg Time:          {all_stats['classifier']['avg_time_ms']}
  - Total Docs:        {all_stats['classifier']['total_classifications']}

Embedding & Indexing:
  - Total Chunks:      {all_stats['retriever']['total_chunks_indexed']}
  - Document Types:    {all_stats['retriever']['document_types']}

LLM Generation:
  - Total Calls:       {all_stats['llm']['total_calls']}
  - Avg Time:          {all_stats['llm']['avg_time_per_call']}

📈 OVERALL METRICS
{'-'*70}
Documents Processed:   {all_stats['processing']['documents_found']}
Total Chunks Created:  {all_stats['processing']['total_chunks']}
Processing Time:       {all_stats['processing']['processing_time']}
Avg Chunk Size:        {all_stats['processing']['avg_chunk_size']} words

{'='*70}
"""

    return report

print("Report generator ready")

Report generator ready


In [40]:
# Define a simple test set for evaluation
TEST_SET = [
    {
        'query': 'What is the loan amount?',
        'expected_doc_type': 'Mortgage Contract',
        'ground_truth_keywords': ['loan amount', 'principal']
    },
    {
        'query': 'What is the monthly salary?',
        'expected_doc_type': 'Pay Slip',
        'ground_truth_keywords': ['gross pay', 'net pay', 'salary']
    },
    {
        'query': 'What are the terms of the agreement?',
        'expected_doc_type': 'Contract',
        'ground_truth_keywords': ['agreement', 'terms and conditions', 'covenant']
    }
]

# Define a simple answer quality evaluation function
def evaluate_answer_quality(test_set, processor):
    """Simple evaluation of answer quality."""
    results = {
        'answer_accuracy': [],
        'citation_rate': [],
        'avg_response_time': [],
        'avg_answer_length': []
    }
    total_answer_length = 0

    print("\nRunning answer quality evaluation...\n")

    for i, test_case in enumerate(test_set):
        query = test_case['query']
        # For simplicity, we won't check ground truth answers, just format and sources
        start_time = time.time()
        result = processor.query(query)
        response_time = time.time() - start_time

        results['avg_response_time'].append(response_time)

        # Check if answer is not empty and contains some text
        is_accurate = len(result.answer.split()) > 5 and "not find relevant information" not in result.answer
        results['answer_accuracy'].append(1 if is_accurate else 0)

        # Check if sources are cited
        has_citation = len(result.sources) > 0
        results['citation_rate'].append(1 if has_citation else 0)

        # Answer length
        answer_length = len(result.answer.split())
        total_answer_length += answer_length

        print(f"Query {i+1}/{len(test_set)}: {query[:50]}...")
        print(f"  Response Time: {response_time:.2f}s, Answer Length: {answer_length} words")
        print(f"  Answer Quality (simple): {'Pass' if is_accurate else 'Fail'}, Cited Sources: {'Yes' if has_citation else 'No'}")
        print()


    avg_results = {
        'answer_accuracy': np.mean(results['answer_accuracy']) * 100,
        'citation_rate': np.mean(results['citation_rate']) * 100,
        'avg_response_time': np.mean(results['avg_response_time']),
        'avg_answer_length': total_answer_length / len(test_set) if test_set else 0
    }

    return avg_results

print("Test set and evaluation functions defined.")

Test set and evaluation functions defined.


In [41]:
"""
Execute this cell AFTER processing your test document.
This will generate all metrics for your PowerPoint.
"""

# First, make sure you have processed a document
if not processor.is_ready:
    print("❌ Please process a document first!")
    print("   1. Upload a PDF in the Gradio interface")
    print("   2. Wait for processing to complete")
    print("   3. Then run this cell")
else:
    print("Starting comprehensive evaluation...\n")

    # Run retrieval evaluation
    print("="*70)
    print("PHASE 1: RETRIEVAL EVALUATION")
    print("="*70)
    retrieval_metrics = evaluate_retrieval_performance(TEST_SET, processor, k=5)

    print("\n" + "="*70)
    print("PHASE 2: ANSWER QUALITY EVALUATION")
    print("="*70)
    answer_metrics = evaluate_answer_quality(TEST_SET, processor)

    # Generate report
    print("\n" + "="*70)
    print("GENERATING FINAL REPORT")
    print("="*70)

    if retrieval_metrics and answer_metrics:
        report = generate_performance_report(retrieval_metrics, answer_metrics, processor)
        print(report)

        # Save to file
        with open('/content/performance_report.txt', 'w') as f:
            f.write(report)
        print("\n✅ Report saved to: /content/performance_report.txt")

        # Create PowerPoint-friendly summary
        print("\n" + "="*70)
        print("COPY THIS TO YOUR POWERPOINT:")
        print("="*70)

        summary = f"""
🧪 RETRIEVAL PERFORMANCE
- Recall@5: {retrieval_metrics['recall_at_k']:.1f}% (target: 80%+)
- Mean Reciprocal Rank: {retrieval_metrics['mrr']:.2f}
- Precision@5: {retrieval_metrics['precision_at_k']:.1f}%
- Hit Rate: {retrieval_metrics['hit_rate']:.1f}%

💊 END-TO-END ACCURACY
- Answer Accuracy: {answer_metrics['answer_accuracy']:.1f}% (evaluated on {len(TEST_SET)} test questions)
- Citation Accuracy: {answer_metrics['citation_rate']:.1f}%

🧑‍🎓 SYSTEM PERFORMANCE
- Average Response Time: {answer_metrics['avg_response_time']:.2f}s
- Classification: {processor.get_all_stats()['classifier']['avg_time_ms']}
- Retrieval Latency: {retrieval_metrics['avg_retrieval_time_ms']:.0f}ms
- LLM Generation: {processor.get_all_stats()['llm']['avg_time_per_call']}
"""
        print(summary)

❌ Please process a document first!
   1. Upload a PDF in the Gradio interface
   2. Wait for processing to complete
   3. Then run this cell
